In [55]:
import json
import os
import re
import time
from datetime import datetime

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager



TON_FILE = "ton.jsonl"                # input (TonAPI)
OUT_FILE = "fragment_final.jsonl"     # output dataset Fragment (JSONL)

HEADLESS = True
MAX_NEW_FROM_TON = None                # None = tutti i nuovi
SLEEP_BETWEEN_ASSETS = 0.4


BASE = "https://fragment.com"



def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def read_jsonl(path):
    if not os.path.exists(path):
        return []
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out


def append_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def fragment_assets_set(fragment_jsonl):
    if not os.path.exists(fragment_jsonl):
        return set()
    s = set()
    with open(fragment_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                o = json.loads(line)
                a = o.get("Asset")
                if isinstance(a, str) and a.startswith("@"):
                    s.add(a)
            except Exception:
                pass
    return s


def ton_assets_from_tonjsonl(ton_records):
    s = set()
    for r in ton_records:
        for a in (r.get("nft_posseduti") or []):
            if isinstance(a, str) and a.startswith("@"):
                s.add(a)
    return sorted(list(s))


def parse_price_value(text):
    if not text:
        return None

    t = str(text).strip().lower()
    if "transferred" in t:
        return None

    m = re.search(r"[\d][\d\s,\.]*", text)
    if not m:
        return None

    num = m.group(0).strip().replace(" ", "")

    if "," in num and "." not in num:
        parts = num.split(",")
        if len(parts) == 2 and len(parts[1]) == 3:
            num = parts[0] + parts[1]
        else:
            num = num.replace(",", ".")

    if "." in num:
        parts = num.split(".")
        if len(parts) == 2 and len(parts[1]) == 3:
            num = parts[0] + parts[1]

    try:
        return float(num)
    except Exception:
        return None


def normalize_dt_from_time_tag(t):
    if t is None:
        return None
    dt = t.get("datetime")
    if isinstance(dt, str) and dt:
        dt = dt.replace("T", " ").replace("Z", "")
        dt = re.split(r"\+|-", dt)[0].strip()
        return dt
    return t.get_text(strip=True)


def wallet_text(a_tag):
    if not a_tag:
        return None

    short = a_tag.find("span", class_="short")
    if short:
        return short.get_text(strip=True)

    head = a_tag.find("span", class_="head")
    tail = a_tag.find("span", class_="tail")
    if head and tail:
        return (head.get_text(strip=True) + tail.get_text(strip=True)).strip()

    return a_tag.get_text(" ", strip=True)


# ---------- selenium  ----------
def build_driver():
    options = webdriver.ChromeOptions()
    if HEADLESS:
        options.add_argument("--headless=new")

    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    options.add_argument("--disable-renderer-backgrounding")
    options.add_argument("--disable-background-timer-throttling")
    options.add_argument("--log-level=3")
    options.add_experimental_option("excludeSwitches", ["enable-logging"])

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.set_page_load_timeout(60)
    return driver


def safe_get(driver, url, tries=3):
    for i in range(tries):
        try:
            driver.get(url)
            WebDriverWait(driver, 20).until(
                lambda d: d.execute_script("return document.readyState") == "complete"
            )
            return True
        except Exception as e:
            print(f"[WARN] get failed ({i+1}/{tries}): {e}")
            time.sleep(1.0)
    return False


def click_load_more(driver, class_name, max_clicks=200):
    for _ in range(max_clicks):
        try:
            btn = driver.find_element(By.CLASS_NAME, class_name)
            if not btn.is_displayed():
                break
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(0.25)
        except Exception:
            break


# ---------- DOM parsing ----------
def find_section_by_h3(soup, title_text):
    for sec in soup.find_all("section", class_="tm-section clearfix"):
        h3 = sec.find("h3", class_="tm-section-header-text")
        if h3 and h3.get_text(strip=True).lower() == title_text.lower():
            return sec
    return None


def parse_table_rows_bid(section):
    out = []
    if not section:
        return out

    table = section.find("table")
    if not table:
        return out

    tbody = table.find("tbody")
    if not tbody:
        return out

    for tr in tbody.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) < 3:
            continue

        price_div = tds[0].find("div", class_=re.compile(r"table-cell-value"))
        time_tag = tds[1].find("time")
        wallet_a = tds[2].find("a", class_="tm-wallet")

        out.append({
            "Bidder": wallet_text(wallet_a),
            "Date of Bid": normalize_dt_from_time_tag(time_tag),
            "Price": parse_price_value(price_div.get_text(" ", strip=True) if price_div else None),
        })

    return out


def parse_table_rows_ownership(section):
    out = []
    if not section:
        return out

    table = section.find("table")
    if not table:
        return out

    tbody = table.find("tbody")
    if not tbody:
        return out

    for tr in tbody.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) < 3:
            continue

        price_div = tds[0].find("div", class_=re.compile(r"table-cell-value"))
        time_tag = tds[1].find("time")
        wallet_a = tds[2].find("a", class_="tm-wallet")

        out.append({
            "Owner": wallet_text(wallet_a),
            "Date of Sale": normalize_dt_from_time_tag(time_tag),
            "Price": parse_price_value(price_div.get_text(" ", strip=True) if price_div else None),
        })

    return out


def parse_highest_bid(soup):
    box = soup.find("div", class_="tm-section-bid-info")
    if not box:
        return None
    td = box.find("td")
    if not td:
        return None
    v = td.find("div", class_=re.compile(r"table-cell-value"))
    return parse_price_value(v.get_text(" ", strip=True) if v else None)


def scrape_asset_page(driver, asset):
    slug = asset.lstrip("@")
    url = f"{BASE}/username/{slug}"

    if not safe_get(driver, url):
        return None

    click_load_more(driver, "js-load-more-orders")
    click_load_more(driver, "js-load-more-owners")

    soup = BeautifulSoup(driver.page_source, "html.parser")

    bid_sec = find_section_by_h3(soup, "Bid History")
    own_sec = find_section_by_h3(soup, "Ownership History")

    bids = parse_table_rows_bid(bid_sec)
    ownership = parse_table_rows_ownership(own_sec)
    highest_bid = parse_highest_bid(soup)

    rec = {
        "Asset": "@" + slug,
        "Scrape date": now_str(),
        "Ownership history": ownership,
        "Bids": bids,
    }

    if highest_bid is not None:
        rec["Highest bid"] = highest_bid

    return rec


def main():
    if not os.path.exists(TON_FILE):
        print("ERRORE: non trovo", TON_FILE)
        return

    ton_records = read_jsonl(TON_FILE)
    all_assets = ton_assets_from_tonjsonl(ton_records)

    known = fragment_assets_set(OUT_FILE)
    missing = [a for a in all_assets if a not in known]

    if MAX_NEW_FROM_TON is not None:
        missing = missing[:MAX_NEW_FROM_TON]

    if not missing:
        print("Nessun nuovo asset da TON.")
        return

    driver = build_driver()
    try:
        for asset in missing:
            rec = scrape_asset_page(driver, asset)
            if rec is None:
                print("[WARN] skip:", asset)
                continue

            append_jsonl(OUT_FILE, rec)
            known.add(asset)

            print("Added:", asset, "| bids:", len(rec["Bids"]), "| ownership:", len(rec["Ownership history"]))
            time.sleep(SLEEP_BETWEEN_ASSETS)

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

Nessun nuovo asset da TON.


In [57]:
import pandas as pd
df = pd.read_json("fragment_final.jsonl", lines=True)
df

,Asset,Scrape date,Ownership history,Bids,Highest bid,Resale
0,@singapore,2026-01-16 16:40:29,[],[{'Bidder': 'UQCUmsKJ9lBZhuHwS6bxcjUO2ddqMVnQy...,15000.0,no
1,@river,2026-01-16 16:40:30,[],[{'Bidder': 'UQDwZBi6t0w3xXSoEm2ZAqrXJaAwJJrSb...,5555.0,no
2,@trial,2026-01-16 16:40:31,[{'Owner': 'UQDRhSGR_e_QyBCNsZhet8VVSJYj1HAHOx...,"[{'Bidder': 'advanced.t.me', 'Date of Bid': 'M...",1500.0,yes
3,@baaa,2026-01-16 16:40:31,[],[{'Bidder': 'UQDt88M4PhcDUAJhKPs5tfwlcmSSjCToW...,5050.0,no
4,@bb9y,2026-01-16 16:40:32,[],[{'Bidder': 'UQAMuJnI_dCt0PDqmEaUeHvEcqIB5xALK...,5050.0,no
...,...,...,...,...,...,...
19859,@userstatus,2026-01-17 16:26:48,[{'Owner': 'UQAaYMrsG5X24l4aBW4nEqDOzwNAHmFXHg...,[],15.0,NaN
19860,@brakence,2026-01-17 16:27:03,[],[{'Bidder': 'UQB8xM5uxOFLS8Rdi5C8Jlp59qacbSm4c...,104.0,no
19861,@hippodrom,2026-01-17 16:27:04,[],"[{'Bidder': 'angee.t.me', 'Date of Bid': 'Jan ...",104.0,no
19862,@rublyovka,2026-01-17 16:27:04,[],[{'Bidder': 'UQAAUiLFUxew0vkuCSZaOqKGjZNeh-USJ...,104.0,no
